# CodiEsp Medical Coding — Google Colab Training

Trains both stages of the pipeline on a T4 GPU:
- **Stage 1 — NER**: Extracts clinical spans from Spanish text (bsc-bio-ehr-es fine-tuned)
- **Stage 2 — Reranker**: Maps extracted spans to ICD-10 codes (cross-encoder)

**Before running:** `Runtime > Change runtime type > T4 GPU`

Then **Runtime > Run all** — everything else is automated.

## 1. Check GPU

In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        "No GPU detected!\n"
        "Go to: Runtime > Change runtime type > Hardware accelerator > T4 GPU"
    )

gpu_name = torch.cuda.get_device_name(0)
vram_gb  = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"GPU  : {gpu_name}")
print(f"VRAM : {vram_gb:.1f} GB")

## 2. Clone Repository

In [ ]:
import os

REPO_URL = "https://github.com/tylerklement/medical-coding.git"
REPO_DIR = "/content/medical-coding"

if os.path.isdir(REPO_DIR):
    print("Repo already cloned — pulling latest changes...")
    !git -C {REPO_DIR} pull
else:
    print(f"Cloning {REPO_URL}...")
    !git clone {REPO_URL} {REPO_DIR}

os.chdir(REPO_DIR)
print(f"Working directory: {os.getcwd()}")
print("Contents:", sorted(os.listdir('.')))

## 3. Install Dependencies

In [ ]:
!pip install -q "transformers>=4.46" datasets seqeval sentence-transformers faiss-cpu accelerate
print("Dependencies ready.")

## 4. Mount Drive & Download CodiEsp Data
Results and data are cached in Google Drive so future runs skip downloads.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import shutil
from pathlib import Path

DRIVE_CACHE = Path('/content/drive/MyDrive/medical-coding-cache')
DATA_DIR    = Path('data')
DRIVE_CACHE.mkdir(parents=True, exist_ok=True)
DATA_DIR.mkdir(parents=True, exist_ok=True)

drive_data = DRIVE_CACHE / 'data'

if (drive_data / 'train' / 'trainX.tsv').exists():
    print("Data found in Drive cache — restoring...")
    if DATA_DIR.exists():
        shutil.rmtree(DATA_DIR)
    shutil.copytree(drive_data, DATA_DIR)
    print("Data restored from Drive cache.")
else:
    print("Downloading CodiEsp from Zenodo (~11 MB)...")
    !python scripts/download_data.py
    print("Caching to Drive for future runs...")
    if drive_data.exists():
        shutil.rmtree(drive_data)
    shutil.copytree(DATA_DIR, drive_data)
    print("Done.")

## 5. Build ICD-10 Index

In [ ]:
import json, shutil
from pathlib import Path

drive_cm  = DRIVE_CACHE / 'icd10cm_descriptions.json'
drive_pcs = DRIVE_CACHE / 'icd10pcs_descriptions.json'
local_cm  = DATA_DIR / 'icd10cm_descriptions.json'
local_pcs = DATA_DIR / 'icd10pcs_descriptions.json'

if drive_cm.exists() and drive_cm.stat().st_size > 10_000:
    shutil.copy(drive_cm,  local_cm)
    shutil.copy(drive_pcs, local_pcs)
    print(f"ICD-10 index restored from Drive.")
    print(f"  CM  codes : {len(json.load(open(local_cm))):,}")
    print(f"  PCS codes : {len(json.load(open(local_pcs))):,}")
else:
    print("Building ICD-10 index from CMS...")
    !python scripts/build_icd10_index.py
    shutil.copy(local_cm,  drive_cm)
    shutil.copy(local_pcs, drive_pcs)
    print("ICD-10 index built and cached to Drive.")

---
## Stage 1: NER Training
Fine-tunes `bsc-bio-ehr-es` to extract clinical spans from Spanish text.

**T4 estimate: ~25-40 min for 5 epochs on the full dataset.**

In [ ]:
# ── NER config ─────────────────────────────────────────────────────
NER_EPOCHS     = 5
NER_BATCH      = 16    # T4 16 GB — fits comfortably
NER_GRAD_ACCUM = 1
NER_LR         = 2e-5
NER_WORKERS    = 2
NER_OUT        = 'models/ner'

# Set to an int to limit articles for a quick smoke-test (None = full)
NER_MAX_TRAIN = None
NER_MAX_DEV   = None
# ───────────────────────────────────────────────────────────────────

import os
os.makedirs(NER_OUT, exist_ok=True)
os.makedirs('outputs', exist_ok=True)

ner_args = (
    f"--no_wandb"
    f" --fp16"
    f" --epochs {NER_EPOCHS}"
    f" --batch_size {NER_BATCH}"
    f" --grad_accum {NER_GRAD_ACCUM}"
    f" --learning_rate {NER_LR}"
    f" --num_workers {NER_WORKERS}"
    f" --output_dir {NER_OUT}"
)
if NER_MAX_TRAIN: ner_args += f" --max_train_samples {NER_MAX_TRAIN}"
if NER_MAX_DEV:   ner_args += f" --max_dev_samples {NER_MAX_DEV}"

print("Command: python train_ner.py", ner_args)
print("=" * 70)
!python train_ner.py {ner_args}

In [ ]:
# Save NER model to Drive
import json, shutil
from pathlib import Path

drive_ner = DRIVE_CACHE / 'models' / 'ner'
local_ner = Path(NER_OUT)

if local_ner.exists():
    drive_ner.parent.mkdir(parents=True, exist_ok=True)
    if drive_ner.exists():
        shutil.rmtree(drive_ner)
    shutil.copytree(local_ner, drive_ner)
    print(f"NER model saved to Drive: {drive_ner}")

results_path = Path('outputs/ner_results.json')
if results_path.exists():
    r = json.load(open(results_path))
    print("\nNER dev-set results:")
    print(f"  Precision : {r.get('eval_precision', 0):.4f}")
    print(f"  Recall    : {r.get('eval_recall',    0):.4f}")
    print(f"  F1        : {r.get('eval_f1',        0):.4f}")

---
## Stage 2: Reranker Training
Fine-tunes a cross-encoder to score `(clinical span, ICD-10 description)` pairs.
Hard negatives are mined with a bi-encoder, making it learn subtle code distinctions.

**T4 estimate: ~20-30 min for 3 epochs.**

In [ ]:
# ── Reranker config ────────────────────────────────────────────────
RR_EPOCHS        = 3
RR_BATCH         = 32   # cross-encoder pairs are lighter than NER windows
RR_NEGATIVES     = 4    # hard negatives per positive example
RR_OUT           = 'models/reranker'
# ───────────────────────────────────────────────────────────────────

os.makedirs(RR_OUT, exist_ok=True)

rr_args = (
    f"--no_wandb"
    f" --epochs {RR_EPOCHS}"
    f" --batch_size {RR_BATCH}"
    f" --negatives_per_pos {RR_NEGATIVES}"
    f" --output_dir {RR_OUT}"
)

print("Command: python train_reranker.py", rr_args)
print("=" * 70)
!python train_reranker.py {rr_args}

In [ ]:
# Save reranker model to Drive
import shutil
from pathlib import Path

drive_rr = DRIVE_CACHE / 'models' / 'reranker'
local_rr = Path(RR_OUT)

if local_rr.exists():
    drive_rr.parent.mkdir(parents=True, exist_ok=True)
    if drive_rr.exists():
        shutil.rmtree(drive_rr)
    shutil.copytree(local_rr, drive_rr)
    print(f"Reranker model saved to Drive: {drive_rr}")
    print(f"Files: {sorted(f.name for f in drive_rr.iterdir())}")
else:
    print("Reranker directory not found — did training complete?")